# Unknown test set, step 2 — recover each sample

Runs the sequential-dilution recovery on each unknown-test sample folder,
carrying its own copy of the network so the whole path from raw spectra to
recovered ML-format table runs in one place.

**Leakage guard.** Recovery is applied independently inside each split, so no
information from the validation, test or unknown-test spectra reaches the
training set.

**Input** — the per-sample folders from step 1, plus the DNA-probe and per-virus
RNA references.

**Output** — per-sample recovered spectra, coefficients, metrics, and an
ML-format CSV per sample.

**Next** — `03_train_eval_recovered.ipynb`.

**Scale factor.** Spectra are multiplied by `SCALE_FACTOR = 400` before the network sees them and the factor is divided out again before anything is written, so every saved spectrum is on the mean-normalized scale. This is a numerical-conditioning choice only: the measured spectrum, the DNA reference and the ground-truth reference are all scaled by the same constant, so the recovered coefficients and every shape-based metric are unchanged.


In [ ]:
"""
UNIFIED EXTRACTION PIPELINE FOR UNKNOWN TEST SAMPLES
=====================================================
Combines Code 1 (Subgroup Builder) + Code 2 (SDE Extraction)
for all 88 unknown test sample folders.
 
Code 4 (merger) is NOT needed here because each sample already
has its ML_format.csv from the earlier Step 0/1/2 pipeline.
After extraction, the extracted spectra are saved in a new
ML_format CSV inside each sample subfolder.
 
Folder naming convention assumed (from Step 0):
    sample{N}_{virus}-{conc_high}-{conc_low}/
        {date}-sample{N}_{virus}-{conc_high}-{conc_low}-ML_format.csv  (original, keep)
        [extraction outputs written here by this script]
"""
 
# ===========================================================================
# SECTION 0: IMPORTS
# ===========================================================================
import os
import re
import gc
import ast
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from itertools import product
 
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
 
from scipy.spatial.distance import cosine
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
 
warnings.filterwarnings('ignore')
from tqdm import tqdm

In [ ]:
# ===========================================================================
# SECTION 1: USER CONFIGURATION  ← only section you need to edit
# ===========================================================================

# Root folder containing all 88 sample subfolders
SAMPLE_ROOT = (
    "/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/"
    "Unknown_test_with_sequential_dilution_01scale_mean"
)

# Output root: one subfolder per sample will be created here,
# each containing only the extracted ML_format CSV.
# Intermediate files (_subgroups, _extraction) are also written here,
# keeping the original SAMPLE_ROOT untouched.
RESULT_ROOT = (
    "/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/"
    "Unknown_test_extracted_results"
)

# Folder containing {virus}-trueRNA.csv and ProbeDNA-trueRNA.csv
RNA_REF_FOLDER = "/home/zhao/Jiaheng Cui/DNA RNA hybridization/data"
 
# Background (DNA probe) reference — fixed filename convention
DNA_REF_FILENAME = "ProbeDNA-trueRNA.csv"
 
# Output date string for the new extracted ML_format CSV
OUTPUT_DATE_STR = "03092026"
 
# Wavenumber range (must match your training data)
WAVENUMBER_START = 400
WAVENUMBER_END   = 1800
 
# Code 1 parameters
K = 10       # Spectra per subgroup
M = 7        # Subgroups per concentration  →  M×M = 49 combinations
RANDOM_SEED = 42
ALLOW_REPLACEMENT = True
 
# Code 2 parameters
SCALE_FACTOR = 400.0
NUM_EPOCHS   = 1000
SAVE_FIGURES = False   # Set True to save PNG figures (slow, large disk usage)

In [ ]:
# ===========================================================================
# SECTION 2: HELPERS — folder/file parsing
# ===========================================================================
 
_folder_re = re.compile(
    r'^sample(\d+)_(.+?)-(\d+(?:\.\d+)?)-(\d+(?:\.\d+)?)$'
)
 
def parse_sample_folder(folder_name):
    """
    Parse 'sample{N}_{virus}-{conc_high}-{conc_low}' → (sample_idx, virus, conc_high, conc_low).
    Returns (None, None, None, None) if the name does not match.
    """
    m = _folder_re.match(folder_name)
    if m is None:
        return None, None, None, None
    return int(m.group(1)), m.group(2), float(m.group(3)), float(m.group(4))
 
 
def get_rna_ref_path(virus_name, rna_ref_folder):
    """Resolve {virus}-trueRNA.csv path; raise FileNotFoundError if missing."""
    path = os.path.join(rna_ref_folder, f"{virus_name}-trueRNA.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"RNA reference not found for virus '{virus_name}': {path}\n"
            f"Please place '{virus_name}-trueRNA.csv' in {rna_ref_folder}"
        )
    return path
 
 
def get_dna_ref_path(rna_ref_folder, dna_ref_filename):
    path = os.path.join(rna_ref_folder, dna_ref_filename)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"DNA reference not found: {path}\n"
            f"Please place '{dna_ref_filename}' in {rna_ref_folder}"
        )
    return path
 

In [ ]:
# ===========================================================================
# SECTION 3: CODE 1 — subgroup builder
# ===========================================================================
 
def build_subgroups(spectra_df, wavenumber_cols, virus_name, conc, M, K,
                    output_folder, random_seed=42):
    """
    Given a DataFrame of spectra for one (virus, conc), create M subgroups
    of K spectra each (force-include strategy with replacement).
 
    Saves:
        {virus}_{conc}_subgroup{i}_avg.csv   ← used by Code 2
        {virus}_{conc}_subgroup{i}.csv        ← raw subgroup (for reference)
 
    Returns list of avg_csv paths.
    """
    if random_seed is not None:
        np.random.seed(random_seed)
 
    total_spectra = len(spectra_df)
 
    # Edge case: fewer spectra than K
    actual_K = min(K, total_spectra)
    actual_M = 1 if actual_K == total_spectra else M
 
    all_subgroup_indices = []
    spectrum_coverage    = {i: [] for i in range(total_spectra)}
 
    for sg_i in range(actual_M):
        if sg_i == 0:
            idx_pool = list(range(total_spectra))
            np.random.shuffle(idx_pool)
            sg_indices = idx_pool[:actual_K]
        else:
            covered = set(j for grp in all_subgroup_indices for j in grp)
            missing = list(set(range(total_spectra)) - covered)
            if missing:
                num_forced = min(len(missing), actual_K)
                forced     = missing[:num_forced]
                extra      = np.random.choice(
                    total_spectra, size=actual_K - num_forced, replace=True
                ).tolist()
                sg_indices = forced + extra
            else:
                sg_indices = np.random.choice(
                    total_spectra, size=actual_K, replace=True
                ).tolist()
 
        all_subgroup_indices.append(sg_indices)
        for idx in sg_indices:
            spectrum_coverage[idx].append(sg_i + 1)
 
    avg_paths = []
    for sg_i, sg_indices in enumerate(all_subgroup_indices):
        sg_data = spectra_df.iloc[sg_indices]
 
        # Raw subgroup CSV
        raw_df = pd.DataFrame({'wavenumber': [int(w) for w in wavenumber_cols]})
        for sp_idx, (_, row) in enumerate(sg_data.iterrows()):
            raw_df[f'spectrum_{sp_idx}'] = row[wavenumber_cols].values
        raw_path = os.path.join(
            output_folder, f"{virus_name}_{conc}_subgroup{sg_i+1}.csv"
        )
        raw_df.to_csv(raw_path, index=False)
 
        # Average CSV
        spec_matrix = raw_df[[c for c in raw_df.columns
                               if c.startswith('spectrum_')]].values
        avg_spectrum = np.mean(spec_matrix, axis=1)
        avg_df = pd.DataFrame({
            'wavenumber':       raw_df['wavenumber'],
            'average_spectrum': avg_spectrum,
        })
        avg_path = os.path.join(
            output_folder, f"{virus_name}_{conc}_subgroup{sg_i+1}_avg.csv"
        )
        avg_df.to_csv(avg_path, index=False)
        avg_paths.append(avg_path)
 
    return avg_paths, actual_M

In [ ]:
# ===========================================================================
# SECTION 4: CODE 2 — SDE neural network
# ===========================================================================
 
class FourierFeatureMapping(nn.Module):
    def __init__(self, num_frequencies=6, include_input=True):
        super().__init__()
        self.num_frequencies = num_frequencies
        self.include_input   = include_input
        self.freq_bands      = (
            2.0 ** torch.arange(0, num_frequencies).float() * np.pi
        )
 
    def forward(self, x):
        out = [x] if self.include_input else []
        for freq in self.freq_bands.to(x.device):
            out.append(torch.sin(freq * x))
            out.append(torch.cos(freq * x))
        return torch.cat(out, dim=-1)
 
 
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim)
        )
        self.activation = nn.ReLU()
 
    def forward(self, x):
        return self.activation(x + self.block(x))
 
 
class SERSDecomposition(nn.Module):
    def __init__(self, input_x_dim=13, hidden_dim=256, z_dim=128):
        super().__init__()
        self.f_input = nn.Sequential(
            nn.Linear(input_x_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),  nn.ReLU(),
        )
        self.f_blocks  = nn.Sequential(ResBlock(hidden_dim), ResBlock(hidden_dim))
        self.f_output  = nn.Sequential(
            nn.Linear(hidden_dim, z_dim), nn.ReLU(), nn.Linear(z_dim, 1)
        )
        self.c_input  = nn.Sequential(nn.Linear(1, hidden_dim), nn.ReLU())
        self.c_blocks = nn.Sequential(
            ResBlock(hidden_dim), ResBlock(hidden_dim), ResBlock(hidden_dim)
        )
        self.c_output = nn.Sequential(nn.Linear(hidden_dim, 2), nn.Softplus())
 
    def forward(self, x_embed, s):
        fx   = self.f_blocks(self.f_input(x_embed))
        f_x  = self.f_output(fx)
        cs   = self.c_blocks(self.c_input(s))
        a_c  = self.c_output(cs)
        return a_c[:, 0:1], a_c[:, 1:2], f_x
 
 
def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
 
 
def flatness_penalty(f_x, threshold, region_length=5):
    f_x    = f_x.view(-1)
    diffs  = f_x[1:] - f_x[:-1]
    sq     = (diffs ** 2).unsqueeze(0).unsqueeze(0)
    kernel = torch.ones(1, 1, region_length, device=f_x.device) / region_length
    avg_sq = F.conv1d(sq, kernel, padding=region_length // 2).squeeze()
    return torch.mean(torch.clamp(threshold - avg_sq, min=0.0))
 
 
def custom_loss(y_true, a_s, b_s, f_x, bg,
                scale_mode=400.0, lambda_penalty=1.0,
                lambda_flat=0.05, threshold=1e-3, apply_flatness=False):
    threshold = threshold * (scale_mode ** 2)
    recon     = a_s * f_x + b_s * bg
    mse_loss  = torch.mean((y_true - recon) ** 2)
    neg_pen   = torch.mean(torch.clamp(-f_x, min=0.0)) * scale_mode
    flat_pen  = (flatness_penalty(f_x, threshold)
                 if apply_flatness else 0.0)
    return mse_loss + lambda_penalty * neg_pen + lambda_flat * flat_pen
 
 
def load_single_spectrum(csv_path):
    df = pd.read_csv(csv_path)
    wn = df.iloc[:, 0].values.astype(np.float32)
    it = df.iloc[:, 1].values.astype(np.float32)
    del df
    return wn, it
 
 
def find_common_length(arrays):
    """Return the safe truncation length across multiple 1-D arrays."""
    min_len = min(len(a) for a in arrays)
    ref = arrays[0][:min_len]
    for a in arrays[1:]:
        if not np.allclose(ref, a[:min_len], rtol=1e-5, atol=1e-5):
            diff = np.where(
                ~np.isclose(ref, a[:min_len], rtol=1e-5, atol=1e-5)
            )[0]
            if len(diff):
                min_len = min(min_len, diff[0])
    return min_len
 
 
def run_extraction(avg_path1, avg_path2, dna_ref_path, rna_ref_path,
                   device, num_epochs, scale_factor):
    """
    Run one SDE extraction on two averaged subgroup spectra.
    Returns the results dict (same structure as original Code 2).
    """
    dna_x, dna_y = load_single_spectrum(dna_ref_path)
    rna_x, rna_y = load_single_spectrum(rna_ref_path)
    x1,    y1    = load_single_spectrum(avg_path1)
    x2,    y2    = load_single_spectrum(avg_path2)
 
    L = find_common_length([dna_x, rna_x, x1, x2])
    common_wn = dna_x[:L].copy()
 
    dna_y, rna_y, y1, y2, x1, x2 = (
        dna_y[:L], rna_y[:L], y1[:L], y2[:L], x1[:L], x2[:L]
    )
    del dna_x, rna_x
 
    dna_norm = dna_y / dna_y.mean();  del dna_y
    rna_norm = rna_y / rna_y.mean();  del rna_y
    y1_norm  = y1    / y1.mean();     del y1
    y2_norm  = y2    / y2.mean();     del y2
 
    x_comb = np.concatenate([x1, x2])
    y_comb = np.concatenate([y1_norm, y2_norm])
    s_comb = np.concatenate([np.zeros(L), np.ones(L)])
    del y1_norm, y2_norm, x1, x2
 
    # --- tensors ---
    s_t  = torch.tensor(s_comb.astype(np.float32).reshape(-1, 1)).to(device)
    bg   = np.tile(dna_norm, 2).reshape(-1, 1)
    x_t  = torch.tensor(x_comb.astype(np.float32).reshape(-1, 1)).to(device)
    y_t  = torch.tensor(y_comb.astype(np.float32).reshape(-1, 1)).to(device)
    bg_t = torch.tensor(bg.astype(np.float32)).to(device)
    del x_comb, y_comb, s_comb, bg
 
    y_t  *= scale_factor
    bg_t *= scale_factor
    x_min, x_max = x_t.min(), x_t.max()
    x_t  = (x_t - x_min) / (x_max - x_min)
 
    fourier   = FourierFeatureMapping(num_frequencies=6).to(device)
    x_embed   = fourier(x_t)
    input_dim = x_embed.shape[1]
 
    model     = SERSDecomposition(input_x_dim=input_dim).to(device)
    model.apply(weights_init)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=1000, gamma=0.5
    )
 
    for epoch in range(num_epochs):
        model.train()
        a_s, b_s, f_x = model(x_embed, s=s_t)
        loss = custom_loss(
            y_t, a_s, b_s, f_x, bg_t,
            scale_mode=scale_factor,
            apply_flatness=(epoch >= 500)
        )
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()
        scheduler.step()
 
    model.eval()
    with torch.no_grad():
        a_s, b_s, f_x = model(x_embed, s=s_t)
 
    del model, optimizer, scheduler, fourier
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
 
    def to_np(t):
        return t.detach().cpu().numpy().reshape(-1)
 
    s_np   = to_np(s_t)
    bg_np  = to_np(bg_t)
    fx_np  = to_np(f_x)
    a_np   = to_np(a_s)
    b_np   = to_np(b_s)
    y_np   = to_np(y_t)
 
    del s_t, bg_t, f_x, a_s, b_s, y_t, x_t, x_embed
 
    mask0 = (s_np == 0.0)
    f0    = fx_np[mask0]
    bg0   = bg_np[mask0]
 
    f_area  = np.trapz(f0,  x=common_wn)
    bg_area = np.trapz(bg0, x=common_wn)
    f_final = f0 * (bg_area / f_area)
    del f0
 
    true_final = rna_norm * scale_factor
 
    r2_ext  = r2_score(true_final, f_final)
    cos_ext = 1 - cosine(true_final, f_final)
    pear_ext, _ = pearsonr(true_final, f_final)
 
    g_final, h_final     = [], []
    r2_r, cos_r, pear_r  = [], [], []
    y_exp_list, recon_list = [], []
 
    for s_val in [0.0, 1.0]:
        mask = (s_np == s_val)
        y_sub = y_np[mask]
        a_v   = a_np[mask][0]
        b_v   = b_np[mask][0]
        a_f   = a_v * (f_area / bg_area)
        b_f   = b_v
        g_final.append(a_f);  h_final.append(b_f)
        recon = a_f * f_final + b_f * bg0
        r2_r.append(r2_score(y_sub, recon))
        cos_r.append(1 - cosine(y_sub, recon))
        p, _ = pearsonr(y_sub, recon);  pear_r.append(p)
        y_exp_list.append(y_sub);  recon_list.append(recon)
 
    del s_np, a_np, b_np, y_np, bg_np, fx_np
    gc.collect()
 
    return {
        'f_final':    f_final,
        'g_final':    np.array(g_final),
        'h_final':    np.array(h_final),
        'common_wn':  common_wn,
        'true_final': true_final,
        'y_exp_list': y_exp_list,
        'recon_list': recon_list,
        'metrics_ext':  {'r2': r2_ext,  'cosine': cos_ext,  'pearson': pear_ext},
        'metrics_recon': {'r2': r2_r,   'cosine': cos_r,    'pearson': pear_r},
    }

In [ ]:
# ===========================================================================
# SECTION 5: PER-SAMPLE ORCHESTRATION
# ===========================================================================
 
def process_one_sample(folder_name, sample_root, result_root,
                       rna_ref_folder, dna_ref_path, device,
                       K, M, random_seed,
                       num_epochs, scale_factor,
                       wavenumber_start, wavenumber_end,
                       output_date_str, save_figures):
    """
    Full pipeline for one sample folder:
      1. Parse folder name → virus, conc_high, conc_low
      2. Load the single combined ML_format CSV; split rows by Conc value
         into df_high (conc_high rows) and df_low (conc_low rows)
      3. Code 1: build M subgroups for each concentration
      4. Code 2: run SDE on all M×M combinations
      5. Write all outputs under result_root/{folder_name}/:
             _subgroups/        — averaged subgroup CSVs
             _extraction/       — extracted_spectra, metrics, coefficients
             {date}-{folder}-extracted_ML_format.csv  ← final deliverable
 
    The original SAMPLE_ROOT folder is never written to.
    Returns a status string: 'ok', 'skip', or 'error: ...'
    """
    folder_path  = os.path.join(sample_root, folder_name)
    output_path  = os.path.join(result_root, folder_name)
    os.makedirs(output_path, exist_ok=True)
 
    # --- parse folder name ---
    sample_idx, virus, conc_high, conc_low = parse_sample_folder(folder_name)
    if virus is None:
        return 'skip'
 
    # --- resolve reference files ---
    try:
        rna_ref_path = get_rna_ref_path(virus, rna_ref_folder)
    except FileNotFoundError as e:
        return f'error: {e}'
 
    # --- find the single combined ML_format CSV ---
    ml_csvs = sorted([
        f for f in os.listdir(folder_path) if f.endswith('ML_format.csv')
    ])
    if len(ml_csvs) == 0:
        return 'error: no ML_format CSV found in sample folder'
    if len(ml_csvs) > 1:
        # Prefer the one whose name does NOT contain 'extracted'
        ml_csvs = [f for f in ml_csvs if 'extracted' not in f.lower()] or ml_csvs
    ml_csv_path = os.path.join(folder_path, ml_csvs[0])
 
    # --- load and split by concentration ---
    wn_cols = [str(w) for w in range(wavenumber_start, wavenumber_end + 1)]
 
    full_df = pd.read_csv(ml_csv_path)
 
    # Parse the Conc column (stored as "[900.0]") into float
    def parse_conc(val):
        try:
            return float(str(val).strip('[]'))
        except ValueError:
            return np.nan
 
    full_df['_conc_float'] = full_df['Conc'].apply(parse_conc)
 
    df_high = full_df[full_df['_conc_float'] == conc_high].copy()
    df_low  = full_df[full_df['_conc_float'] == conc_low ].copy()
 
    if len(df_high) == 0:
        return (f'error: no rows found for conc_high={conc_high} '
                f'(unique concs in CSV: {sorted(full_df["_conc_float"].unique())})')
    if len(df_low) == 0:
        return (f'error: no rows found for conc_low={conc_low} '
                f'(unique concs in CSV: {sorted(full_df["_conc_float"].unique())})')
 
    wn_cols_present = [c for c in wn_cols if c in full_df.columns]
    df_high = df_high[wn_cols_present].reset_index(drop=True)
    df_low  = df_low [wn_cols_present].reset_index(drop=True)
    del full_df
 
    # --- Code 1: subgroup creation (outputs go to result_root) ---
    subgroup_folder = os.path.join(output_path, '_subgroups')
    os.makedirs(subgroup_folder, exist_ok=True)
 
    avg_paths_high, actual_M = build_subgroups(
        df_high, wn_cols_present, virus, conc_high,
        M, K, subgroup_folder, random_seed
    )
    avg_paths_low, _ = build_subgroups(
        df_low, wn_cols_present, virus, conc_low,
        M, K, subgroup_folder, random_seed
    )
    del df_high, df_low
 
    # --- Code 2: extraction ---
    extraction_output_dir = os.path.join(output_path, '_extraction')
    os.makedirs(extraction_output_dir, exist_ok=True)
 
    extracted_cols = {}   # combination_name → f_final array
    metrics_rows   = []
    g_rows, h_rows = [], []
    common_wn      = None
 
    total_combos = actual_M * actual_M
    combo_idx    = 0
 
    for sg1_idx, avg1 in enumerate(avg_paths_high, start=1):
        for sg2_idx, avg2 in enumerate(avg_paths_low, start=1):
            combo_idx += 1
            combo_name = f"{conc_high}sg{sg1_idx}_{conc_low}sg{sg2_idx}"
 
            try:
                results = run_extraction(
                    avg1, avg2,
                    dna_ref_path, rna_ref_path,
                    device, num_epochs, scale_factor
                )
            except Exception as e:
                print(f"    [{combo_idx}/{total_combos}] {combo_name} — ERROR: {e}")
                gc.collect()
                continue
 
            if common_wn is None:
                common_wn = results['common_wn']
 
            extracted_cols[combo_name] = results['f_final'] / SCALE_FACTOR  # undo the x400 conditioning factor once, at the source: every downstream
            metrics_rows.append({
                'combination': combo_name,
                'R2':      results['metrics_ext']['r2'],
                'Cosine':  results['metrics_ext']['cosine'],
                'Pearson': results['metrics_ext']['pearson'],
            })
            g_rows.append({
                'combination': combo_name,
                'g_c1': results['g_final'][0],
                'g_c2': results['g_final'][1],
            })
            h_rows.append({
                'combination': combo_name,
                'h_c1': results['h_final'][0],
                'h_c2': results['h_final'][1],
            })
 
            del results
            gc.collect()
 
    if not extracted_cols:
        return 'error: all combinations failed during extraction'
 
    # --- Save Code-2-style intermediate CSVs under _extraction/ ---
    df_ext = pd.DataFrame({'Wavenumbers': common_wn})
    for cname, spec in extracted_cols.items():
        df_ext[cname] = spec
    df_ext.to_csv(
        os.path.join(extraction_output_dir, 'extracted_spectra_final_epoch.csv'),
        index=False
    )
    pd.DataFrame(metrics_rows).to_csv(
        os.path.join(extraction_output_dir, 'metrics_extracted_vs_true.csv'),
        index=False
    )
    pd.DataFrame(g_rows).to_csv(
        os.path.join(extraction_output_dir, 'g_coefficients_final_epoch.csv'),
        index=False
    )
    pd.DataFrame(h_rows).to_csv(
        os.path.join(extraction_output_dir, 'h_coefficients_final_epoch.csv'),
        index=False
    )
 
    # --- Save final extracted ML_format CSV ---
    # Each combination → one row; wavenumbers interpolated to integer grid.
    int_grid = np.arange(wavenumber_start, wavenumber_end + 1, dtype=float)
 
    ml_rows = []
    for cname, spec in extracted_cols.items():
        spec_interp = np.interp(int_grid, common_wn.astype(float), spec)
        row = {str(int(w)): round(float(v), 6)
               for w, v in zip(int_grid, spec_interp)}
        row['Label'] = f"['{virus}']"
        row['Conc']  = f"[{float(conc_high):.1f}]"
        ml_rows.append(row)
 
    ml_wn_cols = [str(int(w)) for w in int_grid]
    ml_df = pd.DataFrame(ml_rows, columns=ml_wn_cols + ['Label', 'Conc'])
    ml_out_name = f"{output_date_str}-{folder_name}-extracted_ML_format.csv"
    ml_df.to_csv(os.path.join(output_path, ml_out_name), index=False)
 
    return 'ok'

In [ ]:
# ===========================================================================
# SECTION 6: MAIN LOOP
# ===========================================================================
 
if __name__ == '__main__':
 
    # --- Create result root ---
    os.makedirs(RESULT_ROOT, exist_ok=True)
 
    # --- Verify reference files ---
    dna_ref_path = get_dna_ref_path(RNA_REF_FOLDER, DNA_REF_FILENAME)
    print(f"DNA reference : {dna_ref_path}")
    print(f"RNA ref folder: {RNA_REF_FOLDER}")
    print(f"Results root  : {RESULT_ROOT}")
 
    # --- Discover device ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device        : {device}\n")
 
    # --- Collect all sample folders ---
    all_folders = sorted([
        d for d in os.listdir(SAMPLE_ROOT)
        if os.path.isdir(os.path.join(SAMPLE_ROOT, d))
        and d.startswith('sample')
    ])
    print(f"Found {len(all_folders)} sample folders.\n")
 
    # --- Progress log ---
    log_rows = []
    start_time = datetime.now()
 
    pbar = tqdm(all_folders, desc='Extracting samples', unit='sample')
    for folder in pbar:
        t0 = datetime.now()
        pbar.set_postfix({'current': folder[:35]}, refresh=True)
 
        status = process_one_sample(
            folder_name       = folder,
            sample_root       = SAMPLE_ROOT,
            result_root       = RESULT_ROOT,
            rna_ref_folder    = RNA_REF_FOLDER,
            dna_ref_path      = dna_ref_path,
            device            = device,
            K                 = K,
            M                 = M,
            random_seed       = RANDOM_SEED,
            num_epochs        = NUM_EPOCHS,
            scale_factor      = SCALE_FACTOR,
            wavenumber_start  = WAVENUMBER_START,
            wavenumber_end    = WAVENUMBER_END,
            output_date_str   = OUTPUT_DATE_STR,
            save_figures      = SAVE_FIGURES,
        )
 
        elapsed = (datetime.now() - t0).total_seconds()
        # tqdm.write keeps status lines from corrupting the bar
        tqdm.write(f"  {folder}: {status}  ({elapsed:.1f}s)")
 
        _, virus, conc_high, conc_low = parse_sample_folder(folder)
        log_rows.append({
            'folder':    folder,
            'virus':     virus,
            'conc_high': conc_high,
            'conc_low':  conc_low,
            'status':    status,
            'elapsed_s': round(elapsed, 1),
        })
 
    total_elapsed = (datetime.now() - start_time).total_seconds()
    log_df = pd.DataFrame(log_rows)
    log_path = os.path.join(RESULT_ROOT, f"{OUTPUT_DATE_STR}-extraction_pipeline_log.csv")
    log_df.to_csv(log_path, index=False)
 
    ok_count  = (log_df['status'] == 'ok').sum()
    err_count = log_df['status'].str.startswith('error').sum()
    skp_count = (log_df['status'] == 'skip').sum()
 
    print(f"\n{'='*60}")
    print(f"PIPELINE COMPLETE")
    print(f"{'='*60}")
    print(f"  Total samples : {len(all_folders)}")
    print(f"  OK            : {ok_count}")
    print(f"  Errors        : {err_count}")
    print(f"  Skipped       : {skp_count}")
    print(f"  Total time    : {total_elapsed/60:.1f} min")
    print(f"  Log saved     : {log_path}")
 
    if err_count > 0:
        print(f"\nFailed samples:")
        for _, row in log_df[log_df['status'].str.startswith('error')].iterrows():
            print(f"  {row['folder']}: {row['status']}")

#### Unknown test set (variant, concentration) pair summary

In [ ]:
import os
import re
import pandas as pd

SAMPLE_ROOT = (
    "/home/zhao/Jiaheng Cui/DNA RNA hybridization/data/"
    "Unknown_test_with_sequential_dilution_01scale_mean"
)

_folder_re = re.compile(r'^sample\d+_(.+?)-(\d+(?:\.\d+)?)-(\d+(?:\.\d+)?)$')

# Collect (virus, conc_low, conc_high) from every folder name
records = []
for folder in os.listdir(SAMPLE_ROOT):
    m = _folder_re.match(folder)
    if m is None:
        continue
    virus     = m.group(1)
    conc_high = float(m.group(2))
    conc_low  = float(m.group(3))
    # ensure smaller is always first in the pair
    pair = (min(conc_low, conc_high), max(conc_low, conc_high))
    records.append((virus, pair))

# Group by virus, sort pairs by the smaller value, deduplicate
from collections import defaultdict
virus_pairs = defaultdict(list)
for virus, pair in records:
    virus_pairs[virus].append(pair)

rows = []
for virus, pairs in sorted(virus_pairs.items()):
    # deduplicate and sort by the smaller concentration in each pair
    unique_pairs = sorted(set(pairs), key=lambda p: p[0])
    # format as list of [a, b] strings
    conc_str = ', '.join(
        f'[{int(a) if a == int(a) else a}, {int(b) if b == int(b) else b}]'
        for a, b in unique_pairs
    )
    rows.append({'Variant': virus, 'Concentration (PFU/mL)': conc_str})

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(SAMPLE_ROOT, "sample_concentration_summary.csv"), index=False)

#### Model accuracy for each sample

In [ ]:
# ============================================================
# Per-sample spectrum accuracy + visualisations
# Derived from spectrum_level_details.csv — no model re-run needed.
# Uses argmax rule: correct if argmax(prob) == true class
# ============================================================
import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import defaultdict

# ── CONFIG ───────────────────────────────────────────────────
UNKNOWN_RESULT_PATH = (
    f"/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/hybridized model/03172026-Hybridized_unknown_test_weight_10vs1"
)


# UNKNOWN_RESULT_PATH = (
#     f"/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/extracted model/03232026-Extracted_unknown_test_rescaled_weight_100vs1"
# )
# ─────────────────────────────────────────────────────────────

CLASS_NAMES = ['B1', 'B1351', 'B16172', 'BA5', 'EG51', 'JN1', 'SARSCoV2', 'XBB15']

# Load spectrum-level details
spectrum_df = pd.read_csv(
    os.path.join(UNKNOWN_RESULT_PATH, "spectrum_level_details.csv")
)

prob_cols = [f'prob_{c}' for c in CLASS_NAMES]

# Argmax predicted class for each spectrum (gamma-free)
prob_matrix      = spectrum_df[prob_cols].values
argmax_indices   = prob_matrix.argmax(axis=1)
argmax_labels    = [CLASS_NAMES[i] for i in argmax_indices]
spectrum_df['pred_class_argmax'] = argmax_labels

# True class (strip brackets from "['B1']" → 'B1')
spectrum_df['true_class'] = spectrum_df['true_label'].apply(
    lambda s: ast.literal_eval(s)[0]
)
spectrum_df['correct_argmax'] = (
    spectrum_df['pred_class_argmax'] == spectrum_df['true_class']
)

# ── 1. Per-sample accuracy CSV ────────────────────────────────
sample_acc_rows = []
for folder, grp in spectrum_df.groupby('folder', sort=False):
    virus     = grp['true_class'].iloc[0]
    conc_high = grp['true_conc_high'].iloc[0]
    conc_low  = grp['true_conc_low'].iloc[0]
    n_total   = len(grp)
    n_correct = grp['correct_argmax'].sum()
    accuracy  = n_correct / n_total
    sample_acc_rows.append({
        'folder':       folder,
        'variant':      virus,
        'conc_high':    conc_high,
        'conc_low':     conc_low,
        'n_spectra':    n_total,
        'n_correct':    int(n_correct),
        'accuracy':     round(accuracy, 4),
    })

acc_df = pd.DataFrame(sample_acc_rows).sort_values(
    ['variant', 'conc_high']
).reset_index(drop=True)

acc_df.to_csv(
    os.path.join(UNKNOWN_RESULT_PATH, "per_sample_accuracy.csv"), index=False
)
print(acc_df[['folder', 'variant', 'conc_high', 'n_spectra',
              'n_correct', 'accuracy']].to_string(index=False))

# ── 2. Distribution of per-sample accuracy ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(acc_df['accuracy'], bins=20, range=(0, 1),
             color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].axvline(acc_df['accuracy'].mean(), color='red',
                linestyle='--', label=f"Mean = {acc_df['accuracy'].mean():.3f}")
axes[0].set_xlabel("Per-sample accuracy (argmax rule)")
axes[0].set_ylabel("Number of samples")
axes[0].set_title("Distribution of spectrum-level accuracy across 88 samples")
axes[0].legend()

# ECDF — shows the full distribution without binning artefacts
sorted_acc = np.sort(acc_df['accuracy'].values)
ecdf_y     = np.arange(1, len(sorted_acc) + 1) / len(sorted_acc)
axes[1].step(sorted_acc, ecdf_y, color='steelblue', linewidth=2)
axes[1].set_xlabel("Per-sample accuracy (argmax rule)")
axes[1].set_ylabel("Cumulative proportion of samples")
axes[1].set_title("ECDF of per-sample accuracy")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(
    os.path.join(UNKNOWN_RESULT_PATH, "per_sample_accuracy_distribution.png"),
    dpi=150
)
plt.show()

# ── 3. Accuracy vs concentration, coloured by variant ────────
variants       = sorted(acc_df['variant'].unique())
color_map      = cm.get_cmap('tab10', len(variants))
variant_colors = {v: color_map(i) for i, v in enumerate(variants)}

plt.figure(figsize=(8, 5))
for variant in variants:
    sub = acc_df[acc_df['variant'] == variant]
    plt.scatter(
        np.log10(sub['conc_high'].values),
        sub['accuracy'].values,
        color=variant_colors[variant],
        label=variant,
        s=60, edgecolors='grey', linewidths=0.4, alpha=0.85
    )

plt.xlabel("log₁₀(Concentration) — higher conc in pair (PFU/mL)")
plt.ylabel("Per-sample accuracy (argmax rule)")
plt.title("Spectrum-level accuracy vs concentration, coloured by variant")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(UNKNOWN_RESULT_PATH, "accuracy_vs_concentration.png"),
    dpi=150, bbox_inches='tight'
)
plt.show()

# ── Origin export: histogram data + accuracy vs concentration ─

# For the histogram: just one column of 88 accuracy values,
# Origin can bin it itself, or you can use the pre-binned version.
# We save both.

# Raw accuracy column (Origin bins it)
origin_hist = pd.DataFrame({'accuracy': acc_df['accuracy'].values})
origin_hist.to_csv(
    os.path.join(UNKNOWN_RESULT_PATH, "origin_accuracy_raw.csv"),
    index=False
)

# Pre-binned (in case you want to plot as a bar chart in Origin)
counts, bin_edges = np.histogram(acc_df['accuracy'].values, bins=20, range=(0, 1))
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
origin_hist_binned = pd.DataFrame({
    'bin_center': bin_centers,
    'count':      counts,
})
origin_hist_binned.to_csv(
    os.path.join(UNKNOWN_RESULT_PATH, "origin_accuracy_histogram_binned.csv"),
    index=False
)

# For accuracy vs concentration: XY pair per variant
# Format: A_conc(X1), A_acc(Y1), B1351_conc(X2), B1351_acc(Y2), ...
max_len = acc_df['variant'].value_counts().max()
origin_acc_conc = pd.DataFrame()
for variant in sorted(acc_df['variant'].unique()):
    sub = acc_df[acc_df['variant'] == variant].sort_values('conc_high')
    conc_col = sub['conc_high'].values
    acc_col  = sub['accuracy'].values
    # pad shorter variants with NaN so all columns have equal length
    pad = max_len - len(conc_col)
    origin_acc_conc[f'{variant}_conc']     = np.append(conc_col, [np.nan]*pad)
    origin_acc_conc[f'{variant}_accuracy'] = np.append(acc_col,  [np.nan]*pad)

origin_acc_conc.to_csv(
    os.path.join(UNKNOWN_RESULT_PATH, "origin_accuracy_vs_concentration.csv"),
    index=False
)
print("Origin CSVs saved.")

In [ ]:
#### Compare hybridized and extracted accuracy for each sample

In [ ]:
# ============================================================
# Extracted vs Hybridized per-sample accuracy comparison
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ── CONFIG ───────────────────────────────────────────────────
HYBRID_ACC_PATH = os.path.join(
    f"/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/hybridized model/03172026-Hybridized_unknown_test_weight_10vs1",
    "per_sample_accuracy.csv"
)
EXTRACT_ACC_PATH = os.path.join(
    f"/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/extracted model/03232026-Extracted_unknown_test_rescaled_weight_100vs1",
    "per_sample_accuracy.csv"
)
COMPARISON_RESULT_PATH = f"/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/extracted model/03232026-Extracted_unknown_test_rescaled_weight_100vs1/Comparison_with_hybridized"

os.makedirs(COMPARISON_RESULT_PATH, exist_ok=True)
# ─────────────────────────────────────────────────────────────

hybrid_df  = pd.read_csv(HYBRID_ACC_PATH)
extract_df = pd.read_csv(EXTRACT_ACC_PATH)

# Merge on folder so rows are aligned
merged = pd.merge(
    hybrid_df [['folder', 'variant', 'conc_high', 'accuracy']],
    extract_df[['folder', 'accuracy']],
    on='folder', suffixes=('_hybrid', '_extract')
)

# Summary statistics
n_improved  = (merged['accuracy_extract'] > merged['accuracy_hybrid']).sum()
n_same      = (merged['accuracy_extract'] == merged['accuracy_hybrid']).sum()
n_degraded  = (merged['accuracy_extract'] < merged['accuracy_hybrid']).sum()
mean_delta  = (merged['accuracy_extract'] - merged['accuracy_hybrid']).mean()

print(f"Samples improved : {n_improved}/88")
print(f"Samples unchanged: {n_same}/88")
print(f"Samples degraded : {n_degraded}/88")
print(f"Mean Δaccuracy   : {mean_delta:+.4f}  (extract − hybrid)")

# Save comparison table
merged['delta_accuracy'] = merged['accuracy_extract'] - merged['accuracy_hybrid']
merged.to_csv(
    os.path.join(COMPARISON_RESULT_PATH, "accuracy_comparison.csv"), index=False
)

# ── Plot 1: Scatter coloured by log10(concentration) ─────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

log_conc   = np.log10(merged['conc_high'].values)
sc = axes[0].scatter(
    merged['accuracy_hybrid'],
    merged['accuracy_extract'],
    c=log_conc, cmap='RdYlGn', s=60,
    edgecolors='grey', linewidths=0.4, alpha=0.85
)
cbar = plt.colorbar(sc, ax=axes[0])
cbar.set_label("log₁₀(Concentration) (PFU/mL)")

# Diagonal reference line
lims = [0, 1.02]
axes[0].plot(lims, lims, linestyle='--', color='black', linewidth=1, label='y = x')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel("Hybridized model accuracy")
axes[0].set_ylabel("Extracted model accuracy")
axes[0].set_title("Per-sample accuracy: extracted vs hybridized\n(colour = concentration)")
axes[0].legend(fontsize=9)
axes[0].text(0.02, 0.95,
    f"↑ improved: {n_improved}  →↑ same: {n_same}  ↓ degraded: {n_degraded}\n"
    f"Mean Δ = {mean_delta:+.4f}",
    transform=axes[0].transAxes, fontsize=9, va='top',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
)

# ── Plot 2: Scatter coloured by variant ──────────────────────
variants      = sorted(merged['variant'].unique())
color_map     = cm.get_cmap('tab10', len(variants))
variant_color = {v: color_map(i) for i, v in enumerate(variants)}

for variant in variants:
    sub = merged[merged['variant'] == variant]
    axes[1].scatter(
        sub['accuracy_hybrid'],
        sub['accuracy_extract'],
        color=variant_color[variant],
        label=variant, s=60,
        edgecolors='grey', linewidths=0.4, alpha=0.85
    )

axes[1].plot(lims, lims, linestyle='--', color='black', linewidth=1, label='y = x')
axes[1].set_xlim(lims); axes[1].set_ylim(lims)
axes[1].set_xlabel("Hybridized model accuracy")
axes[1].set_ylabel("Extracted model accuracy")
axes[1].set_title("Per-sample accuracy: extracted vs hybridized\n(colour = variant)")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig(
    os.path.join(COMPARISON_RESULT_PATH, "accuracy_comparison_scatter.png"),
    dpi=150, bbox_inches='tight'
)
plt.show()

# ── Plot 3: Delta accuracy vs log10(concentration) ───────────
plt.figure(figsize=(7, 4))
for variant in variants:
    sub = merged[merged['variant'] == variant]
    plt.scatter(
        np.log10(sub['conc_high'].values),
        sub['delta_accuracy'].values,
        color=variant_color[variant],
        label=variant, s=60,
        edgecolors='grey', linewidths=0.4, alpha=0.85
    )
plt.axhline(0, linestyle='--', color='black', linewidth=1)
plt.xlabel("log₁₀(Concentration) (PFU/mL)")
plt.ylabel("Δ accuracy  (extract − hybrid)")
plt.title("Improvement from extraction vs concentration")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(COMPARISON_RESULT_PATH, "delta_accuracy_vs_concentration.png"),
    dpi=150, bbox_inches='tight'
)
plt.show()

# ── Origin export: delta accuracy vs concentration ────────────
# Format: A_conc(X1), A_delta(Y1), B1351_conc(X2), B1351_delta(Y2), ...
max_len_comp = merged['variant'].value_counts().max()
origin_delta = pd.DataFrame()
for variant in sorted(merged['variant'].unique()):
    sub = merged[merged['variant'] == variant].sort_values('conc_high')
    conc_col  = sub['conc_high'].values
    delta_col = sub['delta_accuracy'].values
    pad = max_len_comp - len(conc_col)
    origin_delta[f'{variant}_conc']  = np.append(conc_col,  [np.nan]*pad)
    origin_delta[f'{variant}_delta'] = np.append(delta_col, [np.nan]*pad)

origin_delta.to_csv(
    os.path.join(COMPARISON_RESULT_PATH, "origin_delta_accuracy_vs_concentration.csv"),
    index=False
)
print("Origin delta CSV saved.")